<a href="https://colab.research.google.com/github/fred-creator-creat/etl-powerbi-sql-company/blob/main/Conex%C3%A3o_MySQL_Projeto_Company.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Instala o servidor e o conector
!apt-get update
!apt-get install mysql-server > /dev/null
!pip install mysql-connector-python > /dev/null

# 2. Inicia o serviço
!service mysql start

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu ja

In [2]:
# Comando para mudar o método de autenticação do Root
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY '';"
!mysql -e "FLUSH PRIVILEGES;"

In [3]:
import mysql.connector

# 1. Conectar ao servidor que iniciou na Célula 1 e 2
db = mysql.connector.connect(
  host="localhost",
  user="root",
  password=""
)

cursor = db.cursor()

# 2. Criar o Schema do desafio
cursor.execute("CREATE DATABASE IF NOT EXISTS azure_company")
cursor.execute("USE azure_company")

# 3. Criar a tabela Employee
cursor.execute("DROP TABLE IF EXISTS employee") # Limpar para teste
cursor.execute("""
CREATE TABLE employee(
    Fname varchar(15) not null,
    Minit char(1),
    Lname varchar(15) not null,
    Ssn char(9) not null PRIMARY KEY,
    Bdate date,
    Address varchar(30),
    Sex char(1),
    Salary decimal(10,2),
    Super_ssn char(9),
    Dno int not null
)
""")

# 4. Inserindo os dados
sql_insert_emp = """INSERT INTO employee
(Fname, Minit, Lname, Ssn, Bdate, Address, Sex, Salary, Super_ssn, Dno)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""

dados_employee = [
    ('John', 'B', 'Smith', '123456789', '1965-01-09', '731-Fondren-Houston-TX', 'M', 30000, '333445555', 5),
    ('Franklin', 'T', 'Wong', '333445555', '1955-12-08', '638-Voss-Houston-TX', 'M', 40000, '888665555', 5),
    ('Alicia', 'J', 'Zelaya', '999887777', '1968-01-19', '3321-Castle-Spring-TX', 'F', 25000, '987654321', 4),
    ('Jennifer', 'S', 'Wallace', '987654321', '1941-06-20', '291-Berry-Bellaire-TX', 'F', 43000, '888665555', 4),
    ('Ramesh', 'K', 'Narayan', '666884444', '1962-09-15', '975-Fire-Oak-Humble-TX', 'M', 38000, '333445555', 5),
    ('Joyce', 'A', 'English', '453453453', '1972-07-31', '5631-Rice-Houston-TX', 'F', 25000, '333445555', 5),
    ('Ahmad', 'V', 'Jabbar', '987987987', '1969-03-29', '980-Dallas-Houston-TX', 'M', 25000, '987654321', 4),
    ('James', 'E', 'Borg', '888665555', '1937-11-10', '450-Stone-Houston-TX', 'M', 55000, None, 1)
]

cursor.executemany(sql_insert_emp, dados_employee)
db.commit()

print("Banco 'azure_company' e tabela 'employee' criados e povoados com sucesso!")

Banco 'azure_company' e tabela 'employee' criados e povoados com sucesso!


In [4]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Cria a conexão apontando para o banco (azure_company)
engine = create_engine('mysql+mysqlconnector://root@localhost/azure_company')

# 2. Faz a consulta na tabela de funcionários (employee)
query = "SELECT * FROM employee"
df = pd.read_sql(query, engine)

# 3. Ativa a visualização interativa do Colab
%load_ext google.colab.data_table

# 4. Exibe a tabela com os dados
df

,Fname,Minit,Lname,Ssn,Bdate,Address,Sex,Salary,Super_ssn,Dno
0,John,B,Smith,123456789,1965-01-09,731-Fondren-Houston-TX,M,30000.0,333445555,5
1,Franklin,T,Wong,333445555,1955-12-08,638-Voss-Houston-TX,M,40000.0,888665555,5
2,Joyce,A,English,453453453,1972-07-31,5631-Rice-Houston-TX,F,25000.0,333445555,5
3,Ramesh,K,Narayan,666884444,1962-09-15,975-Fire-Oak-Humble-TX,M,38000.0,333445555,5
4,James,E,Borg,888665555,1937-11-10,450-Stone-Houston-TX,M,55000.0,None,1
5,Jennifer,S,Wallace,987654321,1941-06-20,291-Berry-Bellaire-TX,F,43000.0,888665555,4
6,Ahmad,V,Jabbar,987987987,1969-03-29,980-Dallas-Houston-TX,M,25000.0,987654321,4
7,Alicia,J,Zelaya,999887777,1968-01-19,3321-Castle-Spring-TX,F,25000.0,987654321,4


In [5]:
# 1. Preparar o cursor para as novas tabelas
cursor = db.cursor()

# 2. Criando as Tabelas (Departament, Dept_Locations, Project, Works_on, Dependent)
commands = [
    "DROP TABLE IF EXISTS dependent",
    "DROP TABLE IF EXISTS works_on",
    "DROP TABLE IF EXISTS project",
    "DROP TABLE IF EXISTS dept_locations",
    "DROP TABLE IF EXISTS departament",

    # Tabela Departament
    """CREATE TABLE departament(
        Dname varchar(15) not null,
        Dnumber int not null PRIMARY KEY,
        Mgr_ssn char(9) not null,
        Mgr_start_date date,
        Dept_create_date date
    )""",

    # Tabela Dept_Locations
    """CREATE TABLE dept_locations(
        Dnumber int not null,
        Dlocation varchar(15) not null,
        PRIMARY KEY (Dnumber, Dlocation)
    )""",

    # Tabela Project
    """CREATE TABLE project(
        Pname varchar(15) not null,
        Pnumber int not null PRIMARY KEY,
        Plocation varchar(15),
        Dnum int not null
    )""",

    # Tabela Works_on
    """CREATE TABLE works_on(
        Essn char(9) not null,
        Pno int not null,
        Hours decimal(3,1) not null,
        PRIMARY KEY (Essn, Pno)
    )""",

    # Tabela Dependent
    """CREATE TABLE dependent(
        Essn char(9) not null,
        Dependent_name varchar(15) not null,
        Sex char(1),
        Bdate date,
        Relationship varchar(8),
        PRIMARY KEY (Essn, Dependent_name)
    )"""
]

for command in commands:
    cursor.execute(command)

# 3. Inserindo os Dados em Massa
data_to_insert = {
    "departament": [
        ('Research', 5, '333445555', '1988-05-22','1986-05-22'),
        ('Administration', 4, '987654321', '1995-01-01','1994-01-01'),
        ('Headquarters', 1, '888665555','1981-06-19','1980-06-19')
    ],
    "dept_locations": [
        (1, 'Houston'), (4, 'Stafford'), (5, 'Bellaire'), (5, 'Sugarland'), (5, 'Houston')
    ],
    "project": [
        ('ProductX', 1, 'Bellaire', 5), ('ProductY', 2, 'Sugarland', 5),
        ('ProductZ', 3, 'Houston', 5), ('Computerization', 10, 'Stafford', 4),
        ('Reorganization', 20, 'Houston', 1), ('Newbenefits', 30, 'Stafford', 4)
    ],
    "works_on": [
        ('123456789', 1, 32.5), ('123456789', 2, 7.5), ('666884444', 3, 40.0),
        ('453453453', 1, 20.0), ('453453453', 2, 20.0), ('333445555', 2, 10.0),
        ('333445555', 3, 10.0), ('333445555', 10, 10.0), ('333445555', 20, 10.0),
        ('999887777', 30, 30.0), ('999887777', 10, 10.0), ('987987987', 10, 35.0),
        ('987987987', 30, 5.0), ('987654321', 30, 20.0), ('987654321', 20, 15.0),
        ('888665555', 20, 0.0)
    ],
    "dependent": [
        ('333445555', 'Alice', 'F', '1986-04-05', 'Daughter'),
        ('333445555', 'Theodore', 'M', '1983-10-25', 'Son'),
        ('333445555', 'Joy', 'F', '1958-05-03', 'Spouse'),
        ('987654321', 'Abner', 'M', '1942-02-28', 'Spouse'),
        ('123456789', 'Michael', 'M', '1988-01-04', 'Son'),
        ('123456789', 'Alice', 'F', '1988-12-30', 'Daughter'),
        ('123456789', 'Elizabeth', 'F', '1967-05-05', 'Spouse')
    ]
}

# Loop para inserir em cada tabela
for table, rows in data_to_insert.items():
    placeholders = ", ".join(["%s"] * len(rows[0]))
    sql = f"INSERT INTO {table} VALUES ({placeholders})"
    cursor.executemany(sql, rows)

db.commit()
print("Todas as tabelas foram criadas e povoadas com sucesso!")

Todas as tabelas foram criadas e povoadas com sucesso!


In [6]:
# Exemplo para visualizar a tabela de Project
query_project = "SELECT * FROM project"
df_project = pd.read_sql(query_project, engine)
df_project

,Pname,Pnumber,Plocation,Dnum
0,ProductX,1,Bellaire,5
1,ProductY,2,Sugarland,5
2,ProductZ,3,Houston,5
3,Computerization,10,Stafford,4
4,Reorganization,20,Houston,1
5,Newbenefits,30,Stafford,4


In [7]:
# Exemplo para visualizar a tabela de Departament
query_departament = "SELECT * FROM departament"
df_departament = pd.read_sql(query_departament, engine)
df_departament

,Dname,Dnumber,Mgr_ssn,Mgr_start_date,Dept_create_date
0,Headquarters,1,888665555,1981-06-19,1980-06-19
1,Administration,4,987654321,1995-01-01,1994-01-01
2,Research,5,333445555,1988-05-22,1986-05-22


In [8]:
# Exemplo para visualizar a tabela de Works_on
query_works_on = "SELECT * FROM works_on"
df_works_on = pd.read_sql(query_works_on, engine)
df_works_on

,Essn,Pno,Hours
0,123456789,1,32.5
1,123456789,2,7.5
2,333445555,2,10.0
3,333445555,3,10.0
4,333445555,10,10.0
5,333445555,20,10.0
6,453453453,1,20.0
7,453453453,2,20.0
8,666884444,3,40.0
9,888665555,20,0.0


In [9]:
# Exemplo para visualizar a tabela de Dept_locations
query_dept_locations = "SELECT * FROM dept_locations"
df_dept_locations = pd.read_sql(query_dept_locations, engine)
df_dept_locations

,Dnumber,Dlocation
0,1,Houston
1,4,Stafford
2,5,Bellaire
3,5,Houston
4,5,Sugarland


In [10]:
# Exemplo para visualizar a tabela de Dependent
query_dependent = "SELECT * FROM dependent"
df_dependent = pd.read_sql(query_dependent, engine)
df_dependent

,Essn,Dependent_name,Sex,Bdate,Relationship
0,123456789,Alice,F,1988-12-30,Daughter
1,123456789,Elizabeth,F,1967-05-05,Spouse
2,123456789,Michael,M,1988-01-04,Son
3,333445555,Alice,F,1986-04-05,Daughter
4,333445555,Joy,F,1958-05-03,Spouse
5,333445555,Theodore,M,1983-10-25,Son
6,987654321,Abner,M,1942-02-28,Spouse
